In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle


In [4]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
#Preprocessing the data
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [6]:
#Encoding the categorical data
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

In [7]:
#One hot encoding for the 'Geography' column
ohe = OneHotEncoder()
geography_encoded = ohe.fit_transform(data[['Geography']]).toarray()    
geography_df = pd.DataFrame(geography_encoded, columns=ohe.get_feature_names_out(['Geography']))

In [8]:
geography_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [9]:
#Combining the one-hot encoded 'Geography' columns with the original dataset
data = pd.concat([data.drop('Geography',axis=1), geography_df], axis=1)

In [10]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
#Splitting the dataset into features and target variable
X = data.drop('EstimatedSalary', axis=1)
Y = data['EstimatedSalary']


In [12]:
#Splitting the dataset into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y,test_size=0.2, random_state=42)



In [13]:
#Scale the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
#Save the encoders and scaler for later use
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(le, file)
with open('onehot_encoder.pkl', 'wb') as file:
    pickle.dump(ohe, file)  

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### ANN REGRESION PROBLEM STATEMENT

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [20]:
#Build the model
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='linear')
])

#compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()

/Users/asmitakabra/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [22]:
#Set up tensorboard callback
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [23]:
#Set up early stopping and TensorBoard callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [24]:
#Train the model
history = model.fit(X_train, Y_train, validation_data = (X_test, Y_test), epochs=100,  callbacks=[early_stopping,tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 100145.8047 - mae: 100145.8047 - val_loss: 98573.0625 - val_mae: 98573.0625
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100437.8750 - mae: 100437.8750 - val_loss: 97385.6953 - val_mae: 97385.6953
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 98375.3281 - mae: 98375.3281 - val_loss: 94280.4766 - val_mae: 94280.4766
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 94779.3906 - mae: 94779.3906 - val_loss: 88937.7812 - val_mae: 88937.7812
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 88697.4375 - mae: 88697.4375 - val_loss: 81729.8828 - val_mae: 81729.8828
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 82167.2031 - mae: 82167.2031 - val_loss: 73627.3672 - val_mae: 73627.3672
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 73761.4141 - mae: 73761.4141 - val_loss: 65907.6016 - val_mae: 65907.6016
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/ste

In [25]:
%load_ext tensorboard

In [27]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6008 (pid 33451), started 0:00:08 ago. (Use '!kill 33451' to kill it.)

In [28]:
##Evaluate the model
test_loss, test_mae = model.evaluate(X_test, Y_test)
print(f'Test Loss: {test_loss}, Test MAE: {test_mae}')


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 51164.1523 - mae: 51164.1523
Test Loss: 50366.87109375, Test MAE: 50366.87109375


In [29]:
model.save('salary_regression_model.h5')
